# **Datos Santiago**
---

Notebook de trabajo de la **Fase 2 · Entendimiento de los datos**. Corresponde a la actividad 10 del plan (*Diccionario de datos*): documentar cada variable, su tipo, su dominio y su papel en el proyecto.

Esta primera parte cubre el **componente de preconteo** (`PRE1v_MMV_9999_PRECONTEO.txt`), el MMV (*Mesa a Mesa con Votos*) del conteo rápido de la noche electoral de la primera vuelta presidencial del 31 de mayo de 2026.

## **1. Carga de los datos**
---

La carga reutiliza el código de la sección **2.2 Carga o Adquisición de Datos** del notebook `BSYG_M2U2_fase_1_Entendimiento_del_negocio`, restringida a lo que necesita el diccionario del preconteo: descompresión, verificación de la huella MD5 del preconteo, lectura de ancho fijo, catálogos de partidos y candidatos, y normalización de llaves.

**Antes de ejecutar:** en Colab, suba `MMV_Presidente1V_2026.zip` al panel *Files* (`/content`); en local, el notebook lo busca en `C:\Users\dellphoto\Downloads`. Los archivos se descomprimen en una carpeta `datos` junto al ZIP.

In [5]:
import hashlib
import zipfile
from pathlib import Path
import pandas as pd

# Ruta del ZIP: primero Colab (/content), si no existe se usa la ruta local
RUTAS_ZIP = [Path("/content/MMV_Presidente1V_2026.zip"),
             Path(r"C:\Users\dellphoto\Downloads\MMV_Presidente1V_2026.zip")]
ZIP_ORIGEN = next((r for r in RUTAS_ZIP if r.exists()), None)
if ZIP_ORIGEN is None:
    raise FileNotFoundError(f"No se encontró 'MMV_Presidente1V_2026.zip' en: {[str(r) for r in RUTAS_ZIP]}")

DATA_DIR = ZIP_ORIGEN.parent / "datos"
DATA_DIR.mkdir(parents=True, exist_ok=True)

def buscar_archivo(patron: str, directorio: Path = DATA_DIR) -> Path:
    resultados = list(directorio.rglob(patron))
    if not resultados:
        raise FileNotFoundError(f"No se encontró '{patron}' dentro de {directorio}")
    return resultados[0]

with zipfile.ZipFile(ZIP_ORIGEN) as z:
    z.extractall(DATA_DIR)
with zipfile.ZipFile(buscar_archivo("ARCHIVOSBASICOS_AUDITORES_PRESIDENTE_2026_V3.zip")) as z:
    z.extractall(DATA_DIR)

ARCH_PRECONTEO = buscar_archivo("PRE1v_MMV_9999_PRECONTEO.txt")
DIR_BASICOS    = buscar_archivo("HASH.txt").parent

In [6]:
def calcular_hash(ruta: Path, algoritmo: str = "md5") -> str:
    h = hashlib.new(algoritmo)
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

# MD5 del preconteo tomado del archivo HASH_*.txt de la RNEC
assert calcular_hash(ARCH_PRECONTEO) == "f957d26d6c9ab7eba8e25d14b48047ea", \
    "El preconteo no coincide con la huella publicada"

COLS_PRE   = ["dep", "mun", "zona", "puesto", "mesa", "jal", "comunicado",
              "circ", "partido", "candidato", "votos"]
ANCHOS_PRE = [2, 3, 2, 2, 6, 2, 4, 1, 5, 3, 8]
precon = pd.read_fwf(ARCH_PRECONTEO, widths=ANCHOS_PRE, names=COLS_PRE,
                     dtype=str, encoding="latin-1")
precon["votos"] = precon["votos"].astype("int64")

def leer_catalogo(prefijo: str, anchos, nombres) -> pd.DataFrame:
    ruta = next(DIR_BASICOS.glob(f"{prefijo}_*.TXT"))
    df = pd.read_fwf(ruta, widths=anchos, names=nombres, dtype=str, encoding="latin-1")
    return df.apply(lambda s: s.str.strip())

partidos = leer_catalogo("PARTIDOS", [5, 200], ["partido", "partido_nombre"])
candidatos = leer_catalogo(
    "CANDIDATOS", [3, 1, 2, 3, 2, 5, 3, 1, 50, 50, 15, 1, 2],
    ["corporacion", "circ", "dep", "mun", "comuna", "partido", "candidato",
     "preferente", "nombre", "apellido", "cedula", "genero", "sorteo"])
ESPECIALES = {"996": "VOTO EN BLANCO", "997": "VOTOS NULOS", "998": "VOTOS NO MARCADOS"}

# Normalización de llaves (igual que en la Fase 1)
LLAVE_MESA = ["dep", "mun", "zona", "puesto", "mesa"]
precon["zona"] = precon["zona"].str.zfill(3)
precon["partido"] = precon["partido"].str.zfill(5)

print(f"Preconteo: {len(precon):,} filas x {precon.shape[1]} columnas")
print(f"Mesas: {precon[LLAVE_MESA].drop_duplicates().shape[0]:,}")
precon.head()

Preconteo: 1,479,506 filas x 11 columnas
Mesas: 122,020


,dep,mun,zona,puesto,mesa,jal,comunicado,circ,partido,candidato,votos
0,01,001,001,01,000001,00,9999,0,00009,011,27
1,01,001,001,01,000001,00,9999,0,00015,012,5
2,01,001,001,01,000001,00,9999,0,00020,006,1
3,01,001,001,01,000001,00,9999,0,00021,010,0
4,01,001,001,01,000001,00,9999,0,00022,008,0


## **2. Diccionario de variables del preconteo**
---

El archivo es de **ancho fijo** (38 caracteres por registro, sin encabezado) y está en **formato largo**: cada fila es una combinación *mesa × candidatura* con votos. La estructura de campos sigue el documento oficial *Estructuras Básicas* de la RNEC.

| # | Variable | Posición | Ancho | Descripción | Tipo en pandas | Tipo estadístico | Dominio observado | Papel en el proyecto |
|---|---|---|---|---|---|---|---|---|
| 1 | `dep` | 1–2 | 2 | Código del departamento según la DIVIPOL de la RNEC (no coincide con el código DANE). El código `88` agrupa las mesas del exterior. | Texto | Categórica nominal | 34 códigos (`01`–`72` y `88`) | Llave de mesa; agrupación territorial y partición agrupada por departamento. |
| 2 | `mun` | 3–5 | 3 | Código del municipio dentro del departamento. Solo es único en combinación con `dep`; en el exterior (`dep = 88`) hay 67 códigos. | Texto | Categórica nominal | 309 códigos (`001`–`900`) | Llave de mesa; agrupación territorial. |
| 3 | `zona` | 6–7 | 2 | Código de la zona electoral dentro del municipio. Se normaliza a 3 dígitos (`zfill(3)`) para cruzar con escrutinio, novedades y DIVIPOL. | Texto | Categórica nominal | 52 códigos (`000`–`099`) | Llave de mesa. |
| 4 | `puesto` | 8–9 | 2 | Código del puesto de votación dentro de la zona. Es **alfanumérico**: además de `00`–`99` hay códigos como `A1` o `Z5`. | Texto | Categórica nominal | 208 códigos | Llave de mesa; unidad de comparación entre mesas del mismo puesto. |
| 5 | `mesa` | 10–15 | 6 | Número de la mesa dentro del puesto, con ceros a la izquierda. | Texto | Categórica nominal (identificador) | 228 códigos (`000001`–`000228`) | Llave de mesa; **unidad de análisis**. |
| 6 | `jal` | 16–17 | 2 | Código de comuna/JAL (Junta Administradora Local). No aplica a la elección presidencial. | Texto | Categórica (constante) | Un único valor (`00`) | Sin variabilidad: se descarta como predictor. |
| 7 | `comunicado` | 18–21 | 4 | Número del boletín del preconteo. `9999` identifica el consolidado final. | Texto | Categórica (constante) | Un único valor (`9999`) | Metadato de versión; se descarta como predictor. |
| 8 | `circ` | 22 | 1 | Código de circunscripción. La elección presidencial tiene circunscripción nacional única. | Texto | Categórica (constante) | Un único valor (`0`) | Sin variabilidad: se descarta como predictor. |
| 9 | `partido` | 23–27 | 5 | Código del partido o agrupación que inscribe la candidatura. Se normaliza a 5 dígitos (`zfill(5)`); se traduce con el catálogo `PARTIDOS`. Los códigos especiales 996, 997 y 998 llevan partido `00000`, que no está en el catálogo. | Texto | Categórica nominal | 14 códigos (13 partidos y `00000`) | Identifica la candidatura junto con `candidato`. |
| 10 | `candidato` | 28–30 | 3 | Código de la candidatura: `001`–`013` son los candidatos (catálogo `CANDIDATOS`); `996` voto en blanco, `997` votos nulos y `998` votos no marcados. | Texto | Categórica nominal | 16 códigos | Pivote para pasar a una fila por mesa; base de los indicadores de distribución y concentración del voto. |
| 11 | `votos` | 31–38 | 8 | Número de votos de la candidatura en la mesa según el preconteo. | `int64` | Numérica discreta (conteo ≥ 0) | 0 – 694 (398 valores distintos) | Base de la variable de discrepancia preconteo–escrutinio (regresión) y de los indicadores para la detección de anomalías. |

**Llave de mesa.** Ninguna variable identifica la mesa por sí sola: la llave es la combinación `dep + mun + zona + puesto + mesa`, compartida con el escrutinio y las novedades tras la normalización. La llave de cada fila del preconteo es `llave de mesa + candidato`.

**Texto** significa que la columna se lee con `dtype=str`: pandas la reporta como `object` en versiones anteriores a la 3.0 (Colab) y como `str` desde la 3.0. Los dominios observados se obtuvieron sobre el archivo completo; la celda siguiente los recalcula directamente sobre los datos.

In [7]:
DESCRIPCIONES = {
    "dep":        "Código del departamento (DIVIPOL RNEC)",
    "mun":        "Código del municipio dentro del departamento",
    "zona":       "Código de la zona electoral (normalizado a 3 dígitos)",
    "puesto":     "Código del puesto de votación dentro de la zona",
    "mesa":       "Número de la mesa dentro del puesto",
    "jal":        "Código de comuna/JAL (no aplica a presidencial)",
    "comunicado": "Número de boletín (9999 = consolidado final)",
    "circ":       "Código de circunscripción",
    "partido":    "Código del partido (normalizado a 5 dígitos)",
    "candidato":  "Código de candidatura (001-013, 996, 997, 998)",
    "votos":      "Votos de la candidatura en la mesa",
}
TIPO_ESTADISTICO = {c: "Categórica nominal" for c in COLS_PRE}
TIPO_ESTADISTICO["votos"] = "Numérica discreta"

def resumir_dominio(s: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(s):
        return f"[{s.min():,} – {s.max():,}]"
    valores = sorted(s.unique())
    if len(valores) <= 16:
        return ", ".join(valores)
    return f"{valores[0]} … {valores[-1]}"

diccionario_pre = pd.DataFrame({
    "variable": COLS_PRE,
    "posicion": [f"{ini + 1}-{ini + w}" for ini, w in
                 zip(pd.Series(ANCHOS_PRE).cumsum().shift(fill_value=0), ANCHOS_PRE)],
    "ancho": ANCHOS_PRE,
    "descripcion": [DESCRIPCIONES[c] for c in COLS_PRE],
    "tipo_pandas": precon[COLS_PRE].dtypes.astype(str).values,
    "tipo_estadistico": [TIPO_ESTADISTICO[c] for c in COLS_PRE],
    "nulos": precon[COLS_PRE].isna().sum().values,
    "valores_unicos": precon[COLS_PRE].nunique().values,
    "dominio": [resumir_dominio(precon[c]) for c in COLS_PRE],
})
pd.set_option("display.max_colwidth", 120)
display(diccionario_pre)

,variable,posicion,ancho,descripcion,tipo_pandas,tipo_estadistico,nulos,valores_unicos,dominio
0,dep,1-2,2,Código del departamento (DIVIPOL RNEC),str,Categórica nominal,0,34,01 … 88
1,mun,3-5,3,Código del municipio dentro del departamento,str,Categórica nominal,0,309,001 … 900
2,zona,6-7,2,Código de la zona electoral (normalizado a 3 dígitos),str,Categórica nominal,0,52,000 … 099
3,puesto,8-9,2,Código del puesto de votación dentro de la zona,str,Categórica nominal,0,208,00 … Z5
4,mesa,10-15,6,Número de la mesa dentro del puesto,str,Categórica nominal,0,228,000001 … 000228
5,jal,16-17,2,Código de comuna/JAL (no aplica a presidencial),str,Categórica nominal,0,1,00
6,comunicado,18-21,4,Número de boletín (9999 = consolidado final),str,Categórica nominal,0,1,9999
7,circ,22-22,1,Código de circunscripción,str,Categórica nominal,0,1,0
8,partido,23-27,5,Código del partido (normalizado a 5 dígitos),str,Categórica nominal,0,14,"00000, 00009, 00015, 00020, 00021, 00022, 00026, 01001, 01002, 01003, 01004, 01005, 01006, 03001"
9,candidato,28-30,3,"Código de candidatura (001-013, 996, 997, 998)",str,Categórica nominal,0,16,"001, 002, 003, 004, 005, 006, 007, 008, 009, 010, 011, 012, 013, 996, 997, 998"


### **2.1. Códigos de candidatura y partido**
---

La variable `candidato` es la que da sentido a `votos`. La tabla siguiente traduce cada código con los catálogos `CANDIDATOS` y `PARTIDOS`, agrega los códigos especiales 996, 997 y 998, y muestra cuántas mesas y votos registra cada candidatura en el preconteo.

In [8]:
nombres_cand = (candidatos.assign(nombre_candidatura=candidatos["nombre"] + " " + candidatos["apellido"])
                [["candidato", "nombre_candidatura"]])
nombres_cand = pd.concat([nombres_cand,
                          pd.DataFrame(ESPECIALES.items(), columns=["candidato", "nombre_candidatura"])],
                         ignore_index=True)

codigos_cand = (precon.groupby(["candidato", "partido"], as_index=False)
                .agg(filas=("votos", "size"), votos=("votos", "sum"))
                .merge(nombres_cand, on="candidato", how="left")
                .merge(partidos.assign(partido=partidos["partido"].str.zfill(5)),
                       on="partido", how="left"))
codigos_cand["partido_nombre"] = codigos_cand["partido_nombre"].fillna("(código especial, sin partido)")
codigos_cand["pct_votos"] = (100 * codigos_cand["votos"] / codigos_cand["votos"].sum()).round(2)
codigos_cand["pct_mesas_con_registro"] = (
    100 * codigos_cand["filas"] / precon[LLAVE_MESA].drop_duplicates().shape[0]).round(1)
display(codigos_cand[["candidato", "nombre_candidatura", "partido", "partido_nombre",
                      "filas", "pct_mesas_con_registro", "votos", "pct_votos"]])

,candidato,nombre_candidatura,partido,partido_nombre,filas,pct_mesas_con_registro,votos,pct_votos
0,001,IVÁN CEPEDA CASTRO,00026,MOVIMIENTO POLÍTICO PACTO HISTÓRICO,121967,100.0,9688361,40.40
1,002,CLAUDIA LÓPEZ,01004,CON CLAUDIA IMPARABLES,106109,87.0,225517,0.94
2,003,RAÚL SANTIAGO BOTERO JARAMILLO,01001,ROMPER EL SISTEMA,102972,84.4,206140,0.86
3,004,ABELARDO DE LA ESPRIELLA,01003,DEFENSORES DE LA PATRIA,121597,99.7,10361499,43.21
4,005,ÓSCAR MAURICIO LIZCANO ARANGO,03001,COALICIÓN F.A.M.I.L.I.A,78894,64.7,53839,0.22
5,006,MIGUEL URIBE LONDOÑO,00020,PARTIDO DEMÓCRATA COLOMBIANO,73312,60.1,28657,0.12
6,007,SONDRA MACOLLINS GARVIN PINTO,01002,"SONDRA MACOLLINS, LA ABOGADA DE HIERRO",69947,57.3,19889,0.08
7,008,ROY LEONARDO BARRERAS MONTEALEGRE,00022,PARTIDO POLÍTICO LA FUERZA,67626,55.4,14108,0.06
8,009,CARLOS EDUARDO CAICEDO OMAR,01006,CAICEDO,65201,53.4,12694,0.05
9,010,GUSTAVO MATAMOROS CAMACHO,00021,PARTIDO ECOLOGISTA COLOMBIANO,63261,51.8,5627,0.02


In [9]:
filas_por_mesa = precon.groupby(LLAVE_MESA).size()
print(f"Filas por mesa: mín {filas_por_mesa.min()}, mediana {filas_por_mesa.median():.0f}, máx {filas_por_mesa.max()}")
print(f"Mesas con las 16 candidaturas: {(filas_por_mesa == 16).sum():,} de {len(filas_por_mesa):,}")
print(f"Filas con votos = 0: {(precon['votos'] == 0).sum():,}")
print(f"Combinaciones mesa x candidatura ausentes: {16 * len(filas_por_mesa) - len(precon):,}")
print(f"Duplicados de llave + candidato: {precon.duplicated(LLAVE_MESA + ['candidato']).sum()}")
puestos_alfa = sorted(precon.loc[~precon['puesto'].str.isdigit(), 'puesto'].unique())
print(f"Códigos de puesto alfanuméricos: {len(puestos_alfa)} (p. ej. {', '.join(puestos_alfa[:5])})")

Filas por mesa: mín 1, mediana 13, máx 16
Mesas con las 16 candidaturas: 60,397 de 122,020
Filas con votos = 0: 527,421
Combinaciones mesa x candidatura ausentes: 472,814
Duplicados de llave + candidato: 0
Códigos de puesto alfanuméricos: 108 (p. ej. A1, A2, A3, A4, A5)


### **2.2. Observaciones sobre el preconteo**
---

- **El preconteo no está completo en formato largo.** Tiene 1 479 506 filas, frente a las 1 952 320 del escrutinio (122 020 mesas × 16 candidaturas): cada mesa trae entre 1 y 16 filas. Las filas con cero votos **sí aparecen en algunos casos** (527 421 filas con `votos = 0`), pero otras combinaciones *mesa × candidatura* simplemente no vienen. En ambos casos el valor es **cero votos**: al pasar a una fila por mesa los faltantes deben rellenarse con 0, no tratarse como datos perdidos. La celda siguiente cuantifica ambos fenómenos y la columna `pct_mesas_con_registro` muestra cuánto afecta a cada candidatura.
- **`puesto` no es numérico.** Incluye códigos alfanuméricos (`A1`, `Z5`, …), así que convertirlo a entero fallaría o perdería información.
- **Tres variables son constantes** (`jal`, `comunicado`, `circ`). Documentan el origen del registro pero no aportan información a los modelos.
- **Las llaves son texto.** Los códigos se leen como `object` para conservar los ceros a la izquierda; convertirlos a número rompería el cruce con escrutinio, novedades y DIVIPOL.
- **Los códigos territoriales son jerárquicos.** `mun`, `zona`, `puesto` y `mesa` solo identifican algo dentro de su nivel superior; cualquier conteo o agrupación debe usar la llave compuesta.
- **Candidatos retirados (009 y 013).** En el preconteo conservan sus votos, mientras que en el escrutinio se trasladan a `998`. Esta diferencia debe aislarse al construir la variable de discrepancia.